In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # add project root so src is importable

import pandas as pd
import numpy as np 
from src.config import CUSTOMERS_CLEAN, LOANS_CLEAN, TRANSACTIONS_CLEAN
from src.config import RANDOM_STATE, TEST_SIZE, TRAIN_IDS, TEST_IDS 
from src.config import CUSTOMERS_TRAIN, CUSTOMERS_TEST 

In [2]:
customers = pd.read_parquet(CUSTOMERS_CLEAN)
loans     = pd.read_parquet(LOANS_CLEAN)
transactions = pd.read_parquet(TRANSACTIONS_CLEAN)

In [3]:
merged = loans.merge(customers, on="customer_id", how="left")
merged 

,loan_id,customer_id,disbursed_date,purpose,amount_pkr,term_months,interest_rate_pct,inflow_to_loan_ratio,defaulted,amount_suspect,...,failed_txns_12m,has_savings,savings_balance_pkr,has_insurance,credit_score,churned_12m,age_missing,income_band_missing,tenure_years,is_whale
0,L500000,C107412,2025-01-01,nano_loan,23638.0,3,20.5,0.22,False,False,...,0,0,0.0,0,481,N,0,0,1.92,0
1,L500001,C106708,2025-05-23,nano_loan,23807.0,1,23.0,0.38,False,False,...,0,0,0.0,1,461,N,0,0,1.83,0
2,L500002,C106123,2024-10-25,merchant_advance,277761.0,1,34.2,21.70,True,False,...,1,0,0.0,1,455,Y,0,0,2.00,0
3,L500003,C112525,2024-08-15,nano_loan,12857.0,3,23.9,0.48,False,False,...,1,0,0.0,0,444,N,0,0,1.75,0
4,L500004,C104772,2024-10-22,nano_loan,22631.0,3,21.7,0.75,False,False,...,2,0,0.0,0,465,N,0,0,1.75,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,L507995,C105831,2025-05-26,device_finance,82169.0,1,22.7,2.79,False,False,...,1,0,0.0,0,426,N,0,0,2.17,0
7996,L507996,C110319,2025-01-12,merchant_advance,134329.0,1,26.9,5.46,True,False,...,2,0,0.0,0,428,N,0,0,1.83,0
7997,L507997,C103624,2024-12-31,nano_loan,24820.0,3,32.4,0.91,False,False,...,2,1,27000.0,0,511,N,0,0,3.75,0
7998,L507998,C111817,2025-04-14,emergency,36555.0,1,34.4,0.68,False,False,...,0,0,0.0,0,564,N,0,0,3.50,0


In [4]:
(~loans["customer_id"].isin(customers["customer_id"])).sum()

np.int64(0)

In [5]:
(~transactions["customer_id"].isin(customers["customer_id"])).sum()

np.int64(0)

In [6]:
good = merged[~merged["amount_suspect"]]        # every loan except the 3 broken ones
guess = good["amount_pkr"] / good["avg_monthly_inflow_pkr"]
np.isclose(guess, good["inflow_to_loan_ratio"], atol=0.01).all() 

np.True_

In [7]:
bad = merged[merged["amount_suspect"]]
recovered = bad["inflow_to_loan_ratio"] * bad["avg_monthly_inflow_pkr"]
recovered

Series([], dtype: float64)

In [8]:
np.isclose(recovered, bad["amount_pkr"].abs(), rtol=0.01)      # within 1%

array([], dtype=bool)

In [9]:
merged["amount_pkr"] = merged["amount_pkr"].abs()
(merged["amount_pkr"] < 0).sum()      # should be 0

np.int64(0)

In [10]:
# txn_summary = transactions.groupby("customer_id").agg(
#     total_txns=("txn_count", "sum"),
#     total_value_pkr=("txn_value_pkr", "sum"),
#     avg_monthly_txns=("txn_count", "mean"),
#     avg_monthly_value_pkr=("txn_value_pkr", "mean"),
#     active_months=("month", "nunique"),
# ).reset_index()

In [11]:
# customers = pd.read_parquet(CUSTOMERS_CLEAN)
# customers = customers.merge(txn_summary, on="customer_id", how="left")

In [12]:
# summary_cols = ["total_txns", "total_value_pkr", "avg_monthly_txns", "avg_monthly_value_pkr", "active_months"]
# [c for c in summary_cols if c not in customers.columns]
# #customers[summary_cols] = customers[summary_cols].fillna(0) 

In [13]:
customers = pd.read_parquet(CUSTOMERS_CLEAN)

txn_summary = transactions.groupby("customer_id").agg(
    total_txns=("txn_count", "sum"),
    total_value_pkr=("txn_value_pkr", "sum"),
    avg_monthly_txns_txntbl=("txn_count", "mean"),      # renamed to avoid the clash
    avg_monthly_value_pkr=("txn_value_pkr", "mean"),
    active_months=("month", "nunique"),
).reset_index() 

customers = customers.merge(txn_summary, on="customer_id", how="left")

summary_cols = ["total_txns", "total_value_pkr", "avg_monthly_txns", "avg_monthly_value_pkr", "active_months"]
customers[summary_cols] = customers[summary_cols].fillna(0)

print(customers.columns.tolist())

['customer_id', 'age', 'region', 'city', 'segment_true', 'onboarding_date', 'wallet_tenure_months', 'declared_income_band', 'avg_monthly_inflow_pkr', 'dependents', 'smartphone_user', 'avg_monthly_txns', 'complaints_12m', 'failed_txns_12m', 'has_savings', 'savings_balance_pkr', 'has_insurance', 'credit_score', 'churned_12m', 'age_missing', 'income_band_missing', 'tenure_years', 'is_whale', 'total_txns', 'total_value_pkr', 'avg_monthly_txns_txntbl', 'avg_monthly_value_pkr', 'active_months']


In [14]:
customers[["avg_monthly_txns", "avg_monthly_txns_txntbl"]].describe()

,avg_monthly_txns,avg_monthly_txns_txntbl
count,15000.000000,15000.000000
mean,20.057433,7.093493
std,17.500216,6.230152
min,2.500000,0.545455
25%,9.700000,3.416667
50%,13.400000,4.750000
75%,21.100000,7.583333
max,147.600000,52.583333


In [15]:
# The ordered list
# Load customers_clean.parquet. Confirm age_missing and income_band_missing exist in the columns.
# Add RANDOM_STATE and TEST_SIZE to config.py.
# Build the input to the split: the unique customer_id values, and alongside each one its churned_12m value (for stratification).
# Decide what to do with customers whose churned_12m is NaN — they can't be stratified. Tell me your choice.
# Split that ID list into train_ids and test_ids, using the fixed seed and stratifying on churn.
# Check: the two lengths sum to 15,000, and train is ~80%.
# Check: the overlap between train_ids and test_ids is empty.
# Check: churn rate in train ≈ churn rate in test ≈ overall churn rate.
# Save train_ids and test_ids to disk. Add both paths to config.py.
# Filter the per-loan table by customer_id into loans-train and loans-test.
# Check: default rate in loans-train ≈ loans-test ≈ 13.9%. If far off, change the seed and go back to step 5.
# Where age_missing == 1, set age back to NaN. Same for income_band where income_band_missing == 1.
# Compute the age median and the modal income band using train rows only. Store both numbers.
# Fill the NaNs in both train and test using those two stored numbers.
# Add the two numbers to config.py.

In [16]:
customers.sample(10)

,customer_id,age,region,city,segment_true,onboarding_date,wallet_tenure_months,declared_income_band,avg_monthly_inflow_pkr,dependents,...,churned_12m,age_missing,income_band_missing,tenure_years,is_whale,total_txns,total_value_pkr,avg_monthly_txns_txntbl,avg_monthly_value_pkr,active_months
7881,C107881,34.0,Punjab,Multan,saver,2023-09-23,21.0,25-50k,29800.0,4,...,Y,0,0,1.75,0,32,79010.0,2.666667,6584.166667,12
8910,C108910,26.0,Punjab,Sargodha,merchant,2024-03-02,16.0,25-50k,105300.0,0,...,N,0,1,1.33,0,111,274440.0,9.250000,22870.000000,12
11460,C111460,38.0,Punjab,Sargodha,merchant,2022-08-19,33.0,50-100k,58100.0,2,...,N,0,0,2.75,0,156,325670.0,13.000000,27139.166667,12
12519,C112519,37.0,Punjab,Sargodha,saver,2024-04-05,15.0,50-100k,58300.0,1,...,N,0,0,1.25,0,38,105700.0,3.166667,8808.333333,12
9414,C109414,30.0,Sindh,Hyderabad,saver,2023-07-23,22.0,50-100k,63400.0,6,...,Y,0,0,1.83,0,22,53280.0,2.000000,4843.636364,11
14044,C114044,25.0,KP,Abbottabad,borrower,2024-04-02,15.0,100-250k,124400.0,0,...,N,0,0,1.25,0,39,131160.0,3.250000,10930.000000,12
9458,C109458,34.0,Punjab,Faisalabad,borrower,2020-12-02,55.0,25-50k,27200.0,0,...,N,0,0,4.58,0,56,153820.0,4.666667,12818.333333,12
12197,C112197,43.0,KP,Peshawar,merchant,2024-01-07,17.0,50-100k,48600.0,3,...,N,0,0,1.42,0,281,722030.0,23.416667,60169.166667,12
2245,C102245,38.0,Sindh,Hyderabad,borrower,2018-10-09,81.0,25-50k,30200.0,2,...,N,0,0,6.75,0,58,135370.0,4.833333,11280.833333,12
468,C100468,18.0,Sindh,Hyderabad,borrower,2024-01-17,17.0,<25k,15100.0,3,...,Y,0,0,1.42,0,56,163860.0,4.666667,13655.000000,12


In [17]:
churnnan=customers[customers["churned_12m"].isna()]
churnnan

,customer_id,age,region,city,segment_true,onboarding_date,wallet_tenure_months,declared_income_band,avg_monthly_inflow_pkr,dependents,...,churned_12m,age_missing,income_band_missing,tenure_years,is_whale,total_txns,total_value_pkr,avg_monthly_txns_txntbl,avg_monthly_value_pkr,active_months
10,C100010,25.0,Punjab,Multan,payroll,2023-09-23,20.0,<25k,10100.0,2,...,NaN,0,0,1.67,0,119,374940.0,9.916667,31245.000000,12
70,C100070,26.0,KP,Mardan,saver,2023-07-21,23.0,50-100k,37400.0,0,...,NaN,0,0,1.92,0,39,88090.0,3.250000,7340.833333,12
192,C100192,39.0,Punjab,Faisalabad,merchant,2022-12-19,30.0,<25k,20000.0,2,...,NaN,0,0,2.50,0,162,388150.0,13.500000,32345.833333,12
216,C100216,37.0,Islamabad,Islamabad,payroll,2023-03-31,27.0,50-100k,69300.0,0,...,NaN,0,0,2.25,0,29,82100.0,2.416667,6841.666667,12
230,C100230,45.0,Punjab,Faisalabad,borrower,2021-06-03,49.0,<25k,23200.0,0,...,NaN,0,0,4.08,0,72,152830.0,6.000000,12735.833333,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14525,C114525,39.0,AJK-GB,Gilgit,merchant,2023-05-01,26.0,50-100k,43300.0,1,...,NaN,0,0,2.17,0,155,427020.0,12.916667,35585.000000,12
14544,C114544,33.0,Punjab,Rawalpindi,saver,2023-11-24,19.0,<25k,20100.0,4,...,NaN,0,0,1.58,0,72,247450.0,6.000000,20620.833333,12
14835,C114835,37.0,Islamabad,Islamabad,merchant,2023-12-27,18.0,<25k,13700.0,4,...,NaN,0,0,1.50,0,169,544490.0,14.083333,45374.166667,12
14908,C114908,41.0,Islamabad,Islamabad,payroll,2023-03-20,26.0,25-50k,27900.0,2,...,NaN,0,0,2.17,0,46,95500.0,3.833333,7958.333333,12


In [18]:
default=loans[loans["customer_id"].isin(churnnan["customer_id"])]
default

,loan_id,customer_id,disbursed_date,purpose,amount_pkr,term_months,interest_rate_pct,inflow_to_loan_ratio,defaulted,amount_suspect
10,L500010,C107541,2024-07-05,nano_loan,15922.0,1,32.9,0.14,False,False
55,L500055,C112057,2024-11-04,nano_loan,22029.0,1,28.7,0.90,False,False
106,L500106,C112396,2024-11-30,merchant_advance,258280.0,1,23.8,11.85,True,False
171,L500171,C103058,2024-08-15,device_finance,34494.0,6,32.7,2.43,False,False
189,L500189,C108249,2024-07-27,merchant_advance,377201.0,6,31.1,5.68,True,False
...,...,...,...,...,...,...,...,...,...,...
7741,L507741,C111832,2025-05-07,device_finance,24653.0,1,19.1,0.78,False,False
7774,L507774,C108055,2024-12-04,merchant_advance,340585.0,6,23.6,10.05,True,False
7898,L507898,C106514,2024-07-17,nano_loan,15634.0,3,31.4,0.29,False,False
7964,L507964,C114086,2025-04-16,device_finance,62449.0,12,33.1,1.90,False,False


In [19]:
default["defaulted"].value_counts()

defaulted
False    141
True      23
Name: count, dtype: int64

In [20]:
custom= pd.read_parquet(CUSTOMERS_CLEAN)

from sklearn.model_selection import train_test_split

cust_label=custom[custom["churned_12m"].notna()]
cust_label_train,cust_label_test=train_test_split(
    cust_label["customer_id"],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=cust_label["churned_12m"]
)

no_cust_label=custom[custom["churned_12m"].isna()]
no_cust_label_train, no_cust_label_test =train_test_split(
    no_cust_label["customer_id"],
    test_size=0.20,
    random_state=42,
)

train_values= pd.concat([cust_label_train,no_cust_label_train])
test_values=pd.concat([cust_label_test, no_cust_label_test])

In [21]:
print(len(train_values))

len(test_values) 

12000


3000

In [22]:
train_values.isin(test_values).sum()

np.int64(0)

In [23]:
trains=custom[custom["customer_id"].isin(train_values)]
trains["churned_12m"].value_counts(normalize=True)

churned_12m
N    0.922959
Y    0.077041
Name: proportion, dtype: float64

In [24]:
tests=custom[custom["customer_id"].isin(test_values)]
tests["churned_12m"].value_counts(normalize=True)

churned_12m
N    0.922789
Y    0.077211
Name: proportion, dtype: float64

In [25]:
train_values.to_frame().to_parquet(TRAIN_IDS)
test_values.to_frame().to_parquet(TEST_IDS) 

In [26]:
loan_train=loans[loans["customer_id"].isin(train_values)]
loan_train["defaulted"].value_counts(normalize=True)

defaulted
False    0.860807
True     0.139193
Name: proportion, dtype: float64

In [27]:
tests_loan=loans[loans["customer_id"].isin(test_values)]
tests_loan["defaulted"].value_counts(normalize=True)

defaulted
False    0.861768
True     0.138232
Name: proportion, dtype: float64

In [28]:
print(len(loan_train))

len(tests_loan)

6394


1606

In [29]:
custom.head()

,customer_id,age,region,city,segment_true,onboarding_date,wallet_tenure_months,declared_income_band,avg_monthly_inflow_pkr,dependents,...,failed_txns_12m,has_savings,savings_balance_pkr,has_insurance,credit_score,churned_12m,age_missing,income_band_missing,tenure_years,is_whale
0,C100000,33.0,Sindh,Karachi,payroll,2024-05-14,13.0,25-50k,31700.0,1,...,1,0,0.0,0,442,N,0,0,1.08,0
1,C100001,21.0,Islamabad,Islamabad,merchant,2024-01-07,17.0,<25k,23100.0,0,...,0,1,3300.0,0,435,N,0,0,1.42,0
2,C100002,41.0,KP,Peshawar,borrower,2024-05-08,13.0,25-50k,41000.0,1,...,2,0,0.0,1,426,N,0,0,1.08,0
3,C100003,23.0,Punjab,Lahore,payroll,2024-04-10,15.0,25-50k,15800.0,1,...,0,1,10000.0,0,399,N,0,1,1.25,0
4,C100004,43.0,Punjab,Lahore,payroll,2023-06-28,23.0,<25k,15200.0,1,...,2,0,0.0,1,454,N,0,0,1.92,0


In [30]:
custom.columns

Index(['customer_id', 'age', 'region', 'city', 'segment_true',
       'onboarding_date', 'wallet_tenure_months', 'declared_income_band',
       'avg_monthly_inflow_pkr', 'dependents', 'smartphone_user',
       'avg_monthly_txns', 'complaints_12m', 'failed_txns_12m', 'has_savings',
       'savings_balance_pkr', 'has_insurance', 'credit_score', 'churned_12m',
       'age_missing', 'income_band_missing', 'tenure_years', 'is_whale'],
      dtype='str')

In [31]:
put_customs=custom[custom["customer_id"].isin(train_values)].copy()

In [32]:
put_customs.loc[put_customs["age_missing"]==1,"age"]= np.nan
put_customs.loc[put_customs["income_band_missing"]==1,"declared_income_band"]= np.nan

In [33]:
med=put_customs["age"].median()
put_customs.loc[put_customs["age"].isna(),"age"]= med

In [34]:
mod=put_customs["declared_income_band"].mode()[0]
put_customs.loc[put_customs["declared_income_band"].isna(),"declared_income_band"]= mod

In [35]:
put_customs["declared_income_band"].isna().sum()

np.int64(0)

In [36]:
put_customs["declared_income_band"].dtype

CategoricalDtype(categories=['<25k', '25-50k', '50-100k', '100-250k', '250k+'], ordered=True, categories_dtype=str)

In [37]:
test_customs=custom[custom["customer_id"].isin(test_values)].copy()

In [38]:
test_customs.loc[test_customs["age_missing"]==1,"age"]= np.nan
test_customs.loc[test_customs["income_band_missing"]==1,"declared_income_band"]= np.nan

In [39]:
test_customs.loc[test_customs["age"].isna(),"age"]= med

In [40]:
mod=test_customs["declared_income_band"].mode()[0]
test_customs.loc[test_customs["declared_income_band"].isna(),"declared_income_band"]= mod

In [41]:
test_customs["declared_income_band"].isna().sum()

np.int64(0)

In [42]:
test_customs["declared_income_band"].dtype

CategoricalDtype(categories=['<25k', '25-50k', '50-100k', '100-250k', '250k+'], ordered=True, categories_dtype=str)

In [43]:
put_customs.to_parquet(CUSTOMERS_TRAIN)
test_customs.to_parquet(CUSTOMERS_TEST) 